In [6]:
import tensorflow as tf
from tensorflow.keras.models import model_from_json

# load model
json = 'model_rnn_GRU.120_Dense.5010_LSTMKernelInit.VarianceScaling_DenseKernelInit.lecun_uniformKRl1.0_KRl2.0_recAct.sigmoid_arch.json'
with open(json, 'r') as json_file:
    load = json_file.read()
model = model_from_json(load)

# load weights
weights = 'model_rnn_GRU.120_Dense.5010_LSTMKernelInit.VarianceScaling_DenseKernelInit.lecun_uniformKRl1.0_KRl2.0_recAct.sigmoid_weights.h5'
model.load_weights(weights)

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 15, 6)]           0         
                                                                 
 gru (GRU)                   (None, 120)               46080     
                                                                 
 dense_0 (Dense)             (None, 50)                6050      
                                                                 
 dense_1 (Dense)             (None, 10)                510       
                                                                 
 output_softmax (Dense)      (None, 3)                 33        
                                                                 
Total params: 52673 (205.75 KB)
Trainable params: 52673 (205.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
import numpy as np

# bin = np.binary_repr(num, width=width)

# converts decimal number as shows up in modelsim to actual decimal number
# since verilog implementation uses fixed point and python does not
def fixed_to_dec(num, nfrac):
    return (num / (2 ** nfrac))

# vice versa
def dec_to_fixed(num, nfrac):
    return (num * 2 ** nfrac)

print(np.binary_repr(-1, width=16))
print(fixed_to_dec(-1, 10))
print(fixed_to_dec(np.array([-1, -1]), 10))

1111111111111111
-0.0009765625
[-0.00097656 -0.00097656]


In [8]:
print(dec_to_fixed(-0.56890726, 10))
print(dec_to_fixed(0.00138, 10))
print(fixed_to_dec(903, 10))
print(fixed_to_dec(9613, 10))

-582.56103424
1.41312
0.8818359375
9.3876953125


In [11]:
import tensorflow as tf

# grab GRU layer from overall model
gru = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("gru").output
)

# constant -0.0009765625 input
x = np.full((1, 15, 6), -0.0009765625, dtype=np.float32)

gru_output = gru.predict(x)

print("GRU output shape:", gru_output.shape)
print("GRU output for -1 input:", gru_output)

# constant 0 input
x = np.full((1, 15, 6), 0, dtype=np.float32)

gru_output = gru.predict(x)

print("GRU output shape:", gru_output.shape)
print("GRU output for 0 input:", gru_output)

1/1 [==============================] - 0s 322ms/step
GRU output shape: (1, 120)
GRU output for -1 input: [[-0.56890726 -0.09567672 -0.34913725 -0.99494064  0.3613983   0.19687709
  -0.05769272 -0.14897995  0.1877181  -0.03550649 -0.24532282  0.10750702
  -0.11598816 -0.18015094 -0.16135089 -0.85767335 -0.27906138  0.7703383
  -0.5340335   0.05125267 -0.25609422 -0.25098908  0.4315106  -0.37832123
  -0.20760763  0.7366304  -0.07465351  0.7162866   0.43648556  0.05374935
  -0.70293635  0.7977078   0.28382617  0.27658048 -0.54644835  0.12203787
   0.01980045  0.13664083  0.20438394  0.8159071  -0.27387103 -0.2850782
  -0.6261986  -0.11498982 -0.68637383  0.04770102 -0.1362696   0.09853638
  -0.81951106  0.65336573 -0.50525874 -0.1107319  -0.17950605  0.00516415
  -0.18323974 -0.647145   -0.05634918  0.03898488  0.03517921 -0.11741739
  -0.5549089   0.50507605 -0.02983442 -0.88004863 -0.06596029  0.03512058
   0.6822395  -0.19708467 -0.32967156 -0.45299914  0.52376145  0.25188696
  -0.6143

In [ ]:
import numpy as np

# -----------------------
# set config
# -----------------------
TIMESTEPS = 15
INPUT_SIZE = 6
HIDDEN_SIZE = 120

# -----------------------
# helper functions
# -----------------------
def load_vector(filename):
    return np.loadtxt(filename)

def load_matrix(filename, rows, cols):
    data = np.loadtxt(filename)
    return data.reshape(rows, cols)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# -----------------------
# get GRU weights
# -----------------------

gru = model.layers[1]

kernel, recurrent_kernel, bias = gru.get_weights()

# split kernels
W_z, W_r, W_h = np.split(kernel, 3, axis=1)
U_z, U_r, U_h = np.split(recurrent_kernel, 3, axis=1)

# split bias into input and recurrent
b_input = bias[0]   # (360,)
b_rec   = bias[1]   # (360,)

# split per gate (update, reset, candidate hidden state)
b_z, b_r, b_h = np.split(b_input, 3)
b_z_rec, b_r_rec, b_h_rec = np.split(b_rec, 3)

# -----------------------
# set input
# -----------------------
x = np.zeros((TIMESTEPS, INPUT_SIZE)) # all zeroes

# -----------------------
# initialize hidden state
# -----------------------
h = np.zeros(HIDDEN_SIZE) # more zeroes

# trace storage
z_trace = []
r_trace = []
h_tilde_trace = []
h_trace = []

# -----------------------
# GRU forward pass
# -----------------------
for t in range(TIMESTEPS):
    x_t = x[t]

    # gates
    z = sigmoid(x_t @ W_z + h @ U_z + b_z + b_z_rec)
    r = sigmoid(x_t @ W_r + h @ U_r + b_r + b_r_rec)

    # candidate (reset_after=True)
    h_tilde = np.tanh(
        x_t @ W_h + b_h +
        r * (h @ U_h + b_h_rec)
    )

    # hidden state update
    h = (1 - z) * h_tilde + z * h

    # save traces
    z_trace.append(z)
    r_trace.append(r)
    h_tilde_trace.append(h_tilde)
    h_trace.append(h)

# convert to arrays
z_trace = np.array(z_trace)
r_trace = np.array(r_trace)
h_tilde_trace = np.array(h_tilde_trace)
h_trace = np.array(h_trace)

# -----------------------
# outputs
# -----------------------
print("Final hidden state (first 10 units):")
print(h[:10])

print("\nHidden state evolution (first unit):")
print(h_trace[:, 0])

Final hidden state (first 10 units):
[-0.56835281 -0.09412968 -0.35088817 -0.99496819  0.3640165   0.19295665
 -0.05823543 -0.15016247  0.18848199 -0.0339588 ]

Hidden state evolution (first unit):
[ 0.03352435 -0.01083271 -0.09597818 -0.20979171 -0.31225219 -0.38451959
 -0.43647438 -0.47690711 -0.50939999 -0.53521844 -0.55493816 -0.56869199
 -0.57617964 -0.57659717 -0.56835281]
